# 🌸 Flower Classification using Vision Transformer (ViT)
### With Attention Maps + Grad-CAM (Explainability)


In [ ]:
!pip install -q kagglehub transformers torchvision matplotlib scikit-learn

In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from transformers import ViTForImageClassification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
import kagglehub
path = kagglehub.dataset_download('imsparsh/flowers-dataset')
data_dir = os.path.join(path, 'flowers')
print(os.listdir(data_dir))

In [ ]:
# visualize raw images
classes = os.listdir(data_dir)
plt.figure(figsize=(10,5))
for i,c in enumerate(classes[:5]):
 img = Image.open(os.path.join(data_dir,c,os.listdir(os.path.join(data_dir,c))[0]))
 plt.subplot(1,5,i+1)
 plt.imshow(img); plt.title(c); plt.axis('off')
plt.show()

In [ ]:
transform = transforms.Compose([
 transforms.Resize((224,224)),
 transforms.ToTensor(),
 transforms.Normalize([0.5],[0.5])
])

In [ ]:
dataset = datasets.ImageFolder(data_dir, transform=transform)
train_size = int(0.8*len(dataset))
val_size = len(dataset)-train_size
train_ds, val_ds = random_split(dataset,[train_size,val_size])
train_loader = DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader = DataLoader(val_ds,batch_size=32)
print(dataset.classes)

In [ ]:
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224',num_labels=5)
model.config.output_attentions=True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
import torch.nn as nn, torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(),lr=5e-5)

In [ ]:
losses=[]
for epoch in range(3):
 model.train(); total=0
 for x,y in train_loader:
  x,y=x.to(device),y.to(device)
  out=model(x).logits
  loss=criterion(out,y)
  optimizer.zero_grad(); loss.backward(); optimizer.step()
  total+=loss.item()
 losses.append(total/len(train_loader))
 print('epoch',epoch,'loss',losses[-1])

In [ ]:
plt.plot(losses); plt.title('Loss'); plt.show()

In [ ]:
# evaluation
model.eval(); preds=[]; labels=[]
with torch.no_grad():
 for x,y in val_loader:
  x=x.to(device)
  p=model(x).logits.argmax(1).cpu().numpy()
  preds.extend(p); labels.extend(y.numpy())
print('acc:', np.mean(np.array(preds)==np.array(labels)))

In [ ]:
cm = confusion_matrix(labels,preds)
ConfusionMatrixDisplay(cm,display_labels=dataset.classes).plot(); plt.show()

In [ ]:
# Attention map
def show_attention(img_path):
 img = Image.open(img_path).convert('RGB')
 t = transform(img).unsqueeze(0).to(device)
 out = model(t)
 att = out.attentions[-1][0,:,0,1:].mean(0).reshape(14,14).cpu().numpy()
 att = np.kron(att, np.ones((16,16)))
 plt.imshow(img); plt.imshow(att,alpha=0.5,cmap='jet'); plt.axis('off'); plt.show()

In [ ]:
# Grad-CAM (simple)
def gradcam(img_path):
 img = Image.open(img_path).convert('RGB')
 x = transform(img).unsqueeze(0).to(device)
 x.requires_grad=True
 out = model(x).logits
 pred = out.argmax()
 out[0,pred].backward()
 grad = x.grad[0].mean(0).cpu().numpy()
 plt.imshow(img); plt.imshow(grad,alpha=0.5,cmap='jet'); plt.axis('off'); plt.show()